In [1]:
import pandas as pd
import numpy as np

In [2]:
paths = {
    'online_retail': '../data/online_retail.csv',
    'profiling_table': '../outputs/profiling_table.csv',
    'comparison_stats': '../outputs/comparison table after cleaning.csv',
    'cleaned_dataset': '../data/cleaned dataset.parquet'
}

In [3]:
df_org = pd.read_csv(paths['online_retail'])
df = df_org.copy()

In [4]:
df_org.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom


In [5]:
profiling_table = pd.read_csv(paths['profiling_table'])
profiling_table

,col_name,dtype,n_missing,missing_rate,is_unique,n_unique,sample_values
0,InvoiceNo,str,0,0.00,False,25900,"['536365', '536366', '536367']"
1,StockCode,str,0,0.00,False,4070,"['85123A', '71053', '84406B']"
2,Description,str,1454,0.27,False,4223,"['WHITE HANGING HEART T-LIGHT HOLDER', 'WHITE ..."
3,Quantity,int64,0,0.00,False,722,"[np.int64(6), np.int64(8), np.int64(2)]"
4,InvoiceDate,str,0,0.00,False,23260,"['12/1/10 8:26', '12/1/10 8:28', '12/1/10 8:34']"
5,UnitPrice,float64,0,0.00,False,1630,"[np.float64(2.55), np.float64(3.39), np.float6..."
6,CustomerID,float64,135080,24.93,False,4372,"[np.float64(17850.0), np.float64(13047.0), np...."
7,Country,str,0,0.00,False,38,"['United Kingdom', 'France', 'Australia']"


## Part A Missing & Duplicate Data Strategies
reference: 
1. https://pandas.pydata.org/docs/user_guide/missing_data.html
2. https://pandas.pydata.org/docs/user_guide/duplicates.html 

In [6]:
# copy df for this section only
dfA = df_org.copy()

In [7]:
# Columns with missing values
missing_columns = dfA.columns[dfA.isnull().any()].tolist()
print(f"Columns with missing values: {missing_columns}\n")

Columns with missing values: ['Description', 'CustomerID']



In [8]:
# Rows with duplicated values
duplicated_values = dfA[dfA.duplicated()]
print(f'Rows with duplicated values: {len(duplicated_values)}\nRows of original: {len(df_org)}')

Rows with duplicated values: 5268
Rows of original: 541909


#### 1.1 drop missing value

In [9]:
# for full dataset
df_test_drop = dfA.dropna()

#for a specific column
dfA.dropna(subset=['Description'])

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,12/9/11 12:50,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,12/9/11 12:50,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/11 12:50,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/11 12:50,4.15,12680.0,France


#### 1.2 fill missing value

In [10]:
# with one specific value
df_test_fill_0 = dfA.fillna(0)
df_test_fill_na = dfA.fillna(None) # or `pd.NA` or `np.nan` will remain na
    # for specific column, use new/dupllicated df
df_test_fill_str = dfA.copy()
df_test_fill_str['Description'] = dfA['Description'].fillna('None')

print(f'NA count before: {sum(dfA['Description'].isna())}')
print(f'After fill 0: {sum(df_test_fill_0['Description'].isna())}')
print(f'After fill na: {sum(df_test_fill_na['Description'].isna())}')
print(f'After fill str: {sum(df_test_fill_str['Description'].isna())}')

NA count before: 1454
After fill 0: 0
After fill na: 1454
After fill str: 0


#### 1.3 add missing indicator column

In [11]:
df_missing_indicator = df_org.copy()

# for one column
df_missing_indicator['CustomerID_missing'] = df_missing_indicator['CustomerID'].isnull().astype(int)

# for whole dataset
for col in missing_columns:
    df_missing_indicator[f'{col}_missing'] = df_missing_indicator[col].isnull().astype(int)
df_missing_indicator.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,CustomerID_missing,Description_missing
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom,0,0
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom,0,0
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom,0,0
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom,0,0
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom,0,0


#### 2.1 drop duplicate

In [12]:
# drop row with duplicate
df_drop_duplicate = dfA[~dfA['InvoiceNo'].duplicated()]

print(f'Row count for df {df_drop_duplicate.shape[0]}')

Row count for df 25900


## Part B Cleaning rules

#### 1. remove rows with `Quantity <= 0`

In [13]:
df = df[df['Quantity'] > 0]

#### 2. remove rows with `UnitPrice <= 0`

In [14]:
df = df[df['UnitPrice'] > 0]

#### 3. remove rows with missing `CustomerID`

In [15]:
df = df[df['CustomerID'].notna()]

#### 4. convert `InvoiceDate` to datetime using `errors="coerce"`
reference: https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html

In [16]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors="coerce")
df.shape

/var/folders/0q/ndh_pk0s37n95y6svtjfy7hw0000gp/T/ipykernel_2366/2996831693.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors="coerce")


(397884, 8)

#### 5. remove rows where `InvoiceDate` is NaT

In [17]:
df = df[df['InvoiceDate']!='NaT'] # Does nothing, the value NaT is not a string
# the following does the same thing: keep only the none na values
df = df[~df['InvoiceDate'].isna()]
df = df[df['InvoiceDate'].notna()]
df = df.dropna(subset = ['InvoiceDate'])
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


In [18]:
df_cleaned = df.copy()
df_cleaned.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Part C Required diagnostics (before/after)
Create a comparison table between df_org and df_cleaned. 

In [19]:
# check dtype, missing, and duplicate columns
comparison_stats = pd.DataFrame({
    "org_dtype": df_org.dtypes,
    "cleaned_dtype": df_cleaned.dtypes,
    "org_n_missing": [df_org[col].isna().sum() for col in df_org.columns],
    "cleaned_n_missing": [df_cleaned[col].isna().sum() for col in df_cleaned.columns],
    "org_n_unique": [df_org[col].nunique() for col in df_org.columns],
    "cleaned_n_unique": [df_cleaned[col].nunique() for col in df_cleaned.columns]
}, index=df_org.columns)

# check that negative columns are removed
    # first identify the numeric columns
num_cols = df_org.select_dtypes(include='number').columns
    # use reindex to match full column names
negative_stats = pd.DataFrame({
    "org_n_negative": [(df_org[col] < 0).sum() for col in num_cols],
    "cleaned_n_negative": [(df_cleaned[col] < 0).sum() for col in num_cols]
}, index=num_cols).reindex(df_org.columns)

# concatenate the full summary
full_summary = pd.concat([comparison_stats, negative_stats], axis=1)
full_summary.to_csv(paths['comparison_stats'])
full_summary

,org_dtype,cleaned_dtype,org_n_missing,cleaned_n_missing,org_n_unique,cleaned_n_unique,org_n_negative,cleaned_n_negative
InvoiceNo,str,str,0,0,25900,18532,NaN,NaN
StockCode,str,str,0,0,4070,3665,NaN,NaN
Description,str,str,1454,0,4223,3877,NaN,NaN
Quantity,int64,int64,0,0,722,301,10624.0,0.0
InvoiceDate,str,datetime64[us],0,0,23260,17282,NaN,NaN
UnitPrice,float64,float64,0,0,1630,440,2.0,0.0
CustomerID,float64,float64,135080,0,4372,4338,0.0,0.0
Country,str,str,0,0,38,37,NaN,NaN


In [20]:
len(df_cleaned)

397884

In [21]:
# percentage of rows dropped
pct_dropped = (len(df_org) - len(df_cleaned)) * 100 / len(df_org)
print(f'Percentage of rows dropped: {round(pct_dropped, 2)}%')

Percentage of rows dropped: 26.58%


In [22]:
df_cleaned.to_parquet(paths['cleaned_dataset'], index=False)